In [ ]:
import os

# Check if already installed to avoid re-running
if not os.path.exists("/content/.deps_installed"):

    !pip install --quiet --upgrade pip
    !pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --quiet --no-deps \
        "xformers==0.0.28.post3" \
        "trl<0.19.0" \
        "peft" \
        "accelerate"
    !pip install --quiet "bitsandbytes>=0.46.0"

    # Write flag so we know install is done
    open("/content/.deps_installed", "w").close()
    print("✅ Installation done. Now go to Runtime > Restart session manually.")
else:
    print("✅ Dependencies already installed, skipping.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Installation done. Now go to Runtime > Restart session manually.


In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --upgrade unsloth_zoo "trl>=0.20.0"
!pip install --quiet "bitsandbytes>=0.46.0"
print("Installed. Now Runtime > Restart session.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installed. Now Runtime > Restart session.


In [ ]:
import os, shutil
if os.path.exists("/content/.deps_installed"): os.remove("/content/.deps_installed")
shutil.rmtree("/content/unsloth_compiled_cache", ignore_errors=True)

In [ ]:
from unsloth import FastVisionModel
import torch

# Dynamic Resolution Scaling for Qwen2.5-VL: a higher pixel ceiling keeps more
# visual tokens so the model can read small legend/axis text. 28 = patch(14)*merge(2).
min_pixels = 256  * 28 * 28      # floor   (~200k px)
max_pixels = 1280 * 28 * 28      # ceiling (~1.0M px)  <- your "1280" target

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-VL-7B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

# WIRE the budget into the processor (this is what was missing).
def set_pixel_budget(proc, mn, mx):
    # In current transformers, Qwen2VLImageProcessor.min_pixels/max_pixels are
    # READ-ONLY properties; the real source of truth is image_processor.size
    # = {"shortest_edge": min_pixels, "longest_edge": max_pixels}. smart_resize
    # reads those at preprocess time. We set size first, then try the legacy
    # attributes (harmlessly skipped on versions where they're read-only).
    ip = getattr(proc, "image_processor", proc)
    try:
        ip.size = {"shortest_edge": mn, "longest_edge": mx}
    except Exception as e:
        print("size set failed:", e)
    for obj in (ip, proc):
        for attr, val in (("min_pixels", mn), ("max_pixels", mx)):
            try:
                setattr(obj, attr, val)
            except (AttributeError, TypeError):
                pass   # read-only property -> size{} already handles it
    return ip

_ip = set_pixel_budget(tokenizer, min_pixels, max_pixels)
print(f"✅ pixel budget wired -> size={_ip.size}")

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.2: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.90G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

✅ pixel budget wired -> size={'shortest_edge': 200704, 'longest_edge': 1003520}


In [ ]:

from google.colab import files

print("Please select your kaggle.json file to upload.")
files.upload()
print("kaggle.json uploaded successfully.")

Please select your kaggle.json file to upload.


Saving kaggle.json to kaggle (1).json
kaggle.json uploaded successfully.


In [ ]:
import os

# 1. Setup Kaggle credentials
if not os.path.exists('kaggle.json'):
    print("WARNING: kaggle.json not found. Please upload it!")
else:
    os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()
    print("Downloading requested Kaggle dataset...")
    !kaggle datasets download -d dhivyaraman123/scicap-dataset
    !unzip -q scicap-dataset.zip -d scicap_data

Dataset URL: https://www.kaggle.com/datasets/dhivyaraman123/scicap-dataset
License(s): CC0-1.0
100% 19.1G/19.1G [17:56<00:00, 19.0MB/s]



In [ ]:
import os
import pandas as pd
from datasets import Dataset, Image

csv_path  = '/content/scicap_data/scicap_train.csv'
image_dir = '/content/scicap_data/scicap_images_compressed/share-task-img-mask/arxiv/train'

df = pd.read_csv(csv_path)
df['image'] = df['image'].astype(str).str.strip().map(os.path.basename)

# Vectorized existence check: one directory listing instead of 300k os.path.exists calls
existing = set(os.listdir(image_dir))
df = df[df['image'].isin(existing)].copy()
df['image_path'] = image_dir + os.sep + df['image']

print(f"Found {len(df)} valid image-caption pairs")

# Store ONLY paths. cast_column makes the column lazily decode on access.
full_ds = Dataset.from_dict({
    'image':   df['image_path'].tolist(),
    'caption': df['caption'].astype(str).tolist(),
})
full_ds = full_ds.cast_column('image', Image())

print(full_ds)

Found 333472 valid image-caption pairs
Dataset({
    features: ['image', 'caption'],
    num_rows: 333472
})


In [ ]:
# === [Feature 1 / Stage 1] Detect & BOUND a text-paragraph region (no reading)
# Uses EasyOCR's DETECTOR only (geometry). It locates text; it does NOT read it.
!pip install -q easyocr
import easyocr, numpy as np

_det = easyocr.Reader(['en'], gpu=True)

def detect_text_block(pil_img, bottom_frac=0.55, min_lines=2):
    """Locate a stacked block of text lines in the lower part of the figure.
    Returns (has_block, box) with box=[x1,y1,x2,y2] normalized 0-1000, or None.
    Detector only -- geometry, NOT recognition (nothing is read)."""
    arr = np.asarray(pil_img.convert("RGB"))
    H, W = arr.shape[:2]
    horizontal_list, _ = _det.detect(arr)           # boxes: [x_min,x_max,y_min,y_max]
    boxes = horizontal_list[0] if horizontal_list else []
    lines = []
    for x_min, x_max, y_min, y_max in boxes:
        cy = (y_min + y_max) / 2
        if cy < H * (1 - bottom_frac):              # not in lower band
            continue
        if (x_max - x_min) < W * 0.12:              # too short for prose
            continue
        lines.append((x_min, y_min, x_max, y_max))
    if len(lines) < min_lines:                       # not a paragraph block
        return False, None
    x1 = min(l[0] for l in lines); y1 = min(l[1] for l in lines)
    x2 = max(l[2] for l in lines); y2 = max(l[3] for l in lines)
    return True, [round(x1 / W * 1000), round(y1 / H * 1000),
                  round(x2 / W * 1000), round(y2 / H * 1000)]

def add_text_block(batch):
    flags, boxes = [], []
    for im in batch["image"]:
        ok, b = detect_text_block(im)
        flags.append(ok); boxes.append(b if b else [0, 0, 0, 0])
    return {"has_text_block": flags, "text_block": boxes}

N_PREP = 2000                                        # covers SFT(2000) >= GRPO(500)
full_ds.reset_format()                               # raw full-res PIL for detection
prepared_ds = full_ds.select(range(min(N_PREP, len(full_ds)))).map(
    add_text_block, batched=True, batch_size=8)
print("figures with a detected text block:",
      sum(prepared_ds["has_text_block"]), "/", len(prepared_ds))


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

figures with a detected text block: 675 / 2000


In [ ]:
# === [Feature 1 / Stage 2+3] FIRING LOGIC -- written, disabled by default ===
# When enabled: block-level recognition on the ONE detected crop, then an
# open-source decoder-only LM digests it into a 1-sentence 'context' hint.
FIRE_TEXT_LM = False                          # flip to True to actually fire
TEXT_LM = "Qwen/Qwen2.5-7B-Instruct"          # open-source unidirectional LM

def _crop_norm(pil_img, box):
    W, H = pil_img.size; x1, y1, x2, y2 = box
    return pil_img.convert("RGB").crop((x1 / 1000 * W, y1 / 1000 * H,
                                        x2 / 1000 * W, y2 / 1000 * H))

def build_firing(reader, llm):
    SYS = ("You are given the explanatory paragraph printed beneath a scientific "
           "figure. In ONE sentence, state what the figure shows and its key "
           "variables, to help a vision model caption it. No preamble.")
    def fire(ex):
        if not ex["has_text_block"]:
            return {"context": ""}
        crop = _crop_norm(ex["image"], ex["text_block"])
        # block-level recognition on THIS ONE region only (paragraph=True groups it)
        block = " ".join(reader.readtext(np.asarray(crop), detail=0,
                                         paragraph=True)).strip()
        if not block:
            return {"context": ""}
        msg = [{"role": "system", "content": SYS},
               {"role": "user", "content": block[:2000]}]
        out = llm(msg, max_new_tokens=60, do_sample=False)[0]["generated_text"][-1]["content"]
        return {"context": out.strip()}
    return fire

if FIRE_TEXT_LM:
    from transformers import pipeline
    import torch
    _llm = pipeline("text-generation", model=TEXT_LM,
                    torch_dtype=torch.bfloat16, device_map="auto")
    full_ds.reset_format()
    prepared_ds = prepared_ds.map(build_firing(_det, _llm))   # adds 'context'
    del _llm; torch.cuda.empty_cache()
    print("context generated for",
          sum(bool(c) for c in prepared_ds["context"]), "figures")
else:
    print("Firing OFF -- boxes detected & cached. Set FIRE_TEXT_LM=True to fire.")


Firing OFF -- boxes detected & cached. Set FIRE_TEXT_LM=True to fire.


In [ ]:
#####USE IN FUTURE to avoid print crash instead of  fire the textlm cell below
/* \* ###
# === [Feature 1] FIRE the text-LM stage ===
import torch
from transformers import pipeline

_llm = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                torch_dtype=torch.bfloat16, device_map="auto")

prepared_ds.reset_format()                                # <- not full_ds: the transform lives here
prepared_ds = prepared_ds.map(build_firing(_det, _llm))   # adds 'context'
prepared_ds.reset_format()                                # in case format carried over to map output

del _llm
torch.cuda.empty_cache()

print("context generated for",
      sum(bool(c) for c in prepared_ds["context"]), "/",
      sum(prepared_ds["has_text_block"]), "figures with text blocks")

*\ */###

In [ ]:
# === [Feature 1] FIRE the text-LM stage ===
import torch
from transformers import pipeline
from transformers import BitsAndBytesConfig

FIRE_TEXT_LM = True
TEXT_LM = "Qwen/Qwen2.5-7B-Instruct"
_llm = pipeline("text-generation", model=TEXT_LM,
                model_kwargs={"quantization_config":
                              BitsAndBytesConfig(load_in_4bit=True,
                                                 bnb_4bit_compute_dtype=torch.bfloat16)},
                device_map="auto")
full_ds.reset_format()
prepared_ds = prepared_ds.map(build_firing(_det, _llm))   # adds 'context'

del _llm
torch.cuda.empty_cache()

print("context generated for",
      sum(bool(c) for c in prepared_ds["context"]), "/",
      sum(prepared_ds["has_text_block"]), "figures with text blocks")
print("sample:", next(c for c in prepared_ds["context"] if c))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_ge

KeyError: 'image'

In [ ]:
prepared_ds.reset_format()   # drop the resize transform from this dataset object
print("context generated for",
      sum(bool(c) for c in prepared_ds["context"]), "/",
      sum(prepared_ds["has_text_block"]), "figures with text blocks")
print("sample:", next(c for c in prepared_ds["context"] if c))

context generated for 672 / 675 figures with text blocks
sample: The figure illustrates scenarios where Eve is entangled with the systems measured by Alice and Bob, but still allows for maximal randomness in a single observable per party through rank-1 projective measurements on maximally entangled qubits.


In [ ]:
# === [Feature 2] Detect & bound sub-figure panels (0-1000 normalized) =======
import numpy as np
from PIL import Image as PILImage

def detect_panels(pil_img, min_frac=0.12):
    """Split a (possibly multi-panel) figure into sub-figure boxes via
    whitespace gutters. Returns list of [x1,y1,x2,y2] in 0-1000 norm coords."""
    g = np.asarray(pil_img.convert("L"))
    H, W = g.shape
    ink = g < 245
    col = ink.mean(0); row = ink.mean(1)
    def segments(profile, n):
        on = profile > 0.02; segs = []; s = None
        for i, v in enumerate(on):
            if v and s is None:
                s = i
            if not v and s is not None:
                if (i - s) / n > min_frac:
                    segs.append((s, i))
                s = None
        if s is not None and (n - s) / n > min_frac:
            segs.append((s, n))
        return segs or [(0, n)]
    xs, ys = segments(col, W), segments(row, H)
    boxes = []
    for (y1, y2) in ys:
        for (x1, x2) in xs:
            boxes.append([round(x1 / W * 1000), round(y1 / H * 1000),
                          round(x2 / W * 1000), round(y2 / H * 1000)])
    return boxes

def add_boxes(batch):
    return {"panels": [detect_panels(im) for im in batch["image"]]}

prepared_ds = prepared_ds.map(add_boxes, batched=True, batch_size=16)
print("panels for sample 0:", prepared_ds[0]["panels"])


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

panels for sample 0: [[251, 93, 492, 272], [508, 93, 753, 272]]


In [ ]:
import random

# ── Multi-resolution augmentation (REPLACES the old 224x224 resize) ────────
# The previous transform did img.resize((224, 224)) which destroyed aspect
# ratio AND threw away exactly the small-text detail we wired max_pixels to
# preserve. Instead we jitter the longest side per sample among a few targets
# (your [768, 1024, 1280] idea) as light augmentation. We only ever DOWNSCALE
# huge figures; the processor's min_pixels/max_pixels budget set in Cell 3 is
# still the hard floor/ceiling, so we never blow the token budget.
AUG_LONGEST = [768, 1024, 1280]     # set to [1280] for fixed high-res, no aug

def _resize_longest(img, target):
    img = img.convert("RGB")
    w, h = img.size
    scale = target / max(w, h)
    if scale < 1.0:                              # only shrink oversized figures
        img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))))
    return img

def transform(batch):
    target = random.choice(AUG_LONGEST)          # per-access resolution jitter
    images = [_resize_longest(img, target) for img in batch['image']]
    out = dict(batch)
    out['image'] = images
    return out

full_ds.set_transform(transform)

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    full_ds,
    batch_size=32,
    num_workers=4,        # parallel image decoding overlaps with GPU compute
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [ ]:
# === [Feature 3] Unsupervised domain-adaptive pretraining (BEFORE SFT) ======
# Causal-LM (next-token) over the scientific caption corpus, text-only, no
# instruction template. Adapts the language tower to scientific writing style
# before instruction SFT teaches the captioning task. Trains the same LoRA
# adapters from Cell 3, so SFT continues on top of this.
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset as HFDataset
from unsloth import is_bf16_supported

text_tok = getattr(tokenizer, "tokenizer", tokenizer)   # underlying text tokenizer
if text_tok.pad_token is None:
    text_tok.pad_token = text_tok.eos_token

DAPT_N = 5000
full_ds.reset_format()
corpus = [c.strip() for c in full_ds["caption"][:DAPT_N] if c and c.strip()]

def _tok(b):
    return text_tok(b["text"], truncation=True, max_length=512)

dapt_ds = HFDataset.from_dict({"text": corpus}).map(
    _tok, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(text_tok, mlm=False)

FastVisionModel.for_training(model)
dapt_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="outputs/dapt_qwen2_scicap",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=5e-5,                 # lower than the 2e-4 used for SFT
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        optim="adamw_8bit",
        bf16=is_bf16_supported(),
        fp16=not is_bf16_supported(),
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    ),
    train_dataset=dapt_ds,
    data_collator=collator,
)
print("Unsupervised DAPT (causal LM on captions) before SFT ...")
dapt_trainer.train()
print("DAPT done -- proceeding to SFT on the same adapter")
# NOTE: this trains the language-tower LoRA via text-only causal LM. If your
# Unsloth build's patched forward complains about missing pixel_values, fall
# back to image-grounded continued pretraining (image + caption, no instruction).


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsupervised DAPT (causal LM on captions) before SFT ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 313
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with g

Step,Training Loss
10,3.312674
20,3.105812
30,3.183869
40,3.039402
50,3.178823
60,3.025810
70,2.920790
80,3.025560
90,3.030030
100,3.077219


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!
DAPT done -- proceeding to SFT on the same adapter


In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted (or use Colab Secrets, see below)

In [ ]:
repo = "shemalfoy/qwen2-vl-unsupervised" # You can change this to a new repository name if needed

model.push_to_hub(repo)
tokenizer.push_to_hub(repo)
print(f"Model and tokenizer pushed to Hugging Face Hub at: {repo}")

README.md:   0%|          | 0.00/595 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 54.9kB /  206MB            

Saved model to https://huggingface.co/shemalfoy/qwen2-vl-unsupervised


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp_ttt6ydf/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp_ttt6ydf/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Model and tokenizer pushed to Hugging Face Hub at: shemalfoy/qwen2-vl-unsupervised


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from torch.utils.data import Dataset as TorchDataset
from transformers import TrainerCallback
import json

# >>> Run huggingface_hub.login() first (Cell 10) so pushes are authenticated <
HF_REPO    = "shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm"   # adapter repo to push to
PUSH_EVERY = 100                                         # push to Hub every N steps


class SciCapDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset           # lazy Image feature -> PIL on access

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        s = self.data[idx]
        ctx    = s.get("context", "")        # Feature 1: text-LM hint (empty until firing)
        panels = s.get("panels", [])         # Feature 2: sub-figure boxes (0-1000 norm)

        # Prompt: ask for sequential region localization+caption, THEN a summary.
        user_text = ("First localize each sub-figure as a JSON list of "
                     "{\"bbox_2d\":[x1,y1,x2,y2], \"caption\":...}, coords "
                     "normalized 0-1000, then give an overall 'Summary:' of the "
                     "whole figure.")
        if ctx:
            user_text = f"Context read from the figure: {ctx}\n\n" + user_text

        # Target: regions first (sequential), then the global caption as Summary.
        region_json = json.dumps(
            [{"bbox_2d": b, "caption": f"sub-figure {i+1}"}
             for i, b in enumerate(panels)], ensure_ascii=False)
        target = f"Regions:\n{region_json}\n\nSummary: {s['caption']}"

        return {
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image": s["image"]},
                    {"type": "text",  "text": user_text},
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": target},
                ]},
            ]
        }


class PushAdapterToHubCallback(TrainerCallback):
    """Persist ONLY by pushing the LoRA adapter to the Hub. With save_strategy="no"
    nothing is checkpointed locally, so peak disk use is just the small transient
    adapter (~tens of MB) written during each upload -- not the multi-GB optimizer
    checkpoints a save_steps run would leave on disk."""
    def __init__(self, repo, every):
        self.repo, self.every = repo, every

    def _push(self, tag):
        try:
            model.push_to_hub(self.repo, commit_message=tag)       # adapter only (~tens of MB)
            tokenizer.push_to_hub(self.repo, commit_message=tag)   # processor config (small)
            print(f"  ^ pushed adapter to {self.repo}  [{tag}]")
        except Exception as e:
            print(f"  ! Hub push failed [{tag}]: {e}  (training continues)")

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % self.every == 0:
            self._push(f"step-{state.global_step}")

    def on_train_end(self, args, state, control, **kwargs):
        self._push("final")


# Feature 1/2: train on the PREPARED slice (text_block + panels, and 'context'
# once firing is enabled). set_transform re-applies the Cell 7 augmentation.
prepared_ds.set_transform(transform)
train_torch_dataset = SciCapDataset(prepared_ds)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,                                  # this is the processor in Unsloth
    # "max" = NO pre-resize; let the processor size{} budget from Cell 3 govern.
    data_collator=UnslothVisionDataCollator(model, tokenizer, resize="max"),
    train_dataset=train_torch_dataset,
    callbacks=[PushAdapterToHubCallback(HF_REPO, PUSH_EVERY)],
    args=SFTConfig(
        output_dir="outputs/sft_qwen2_scicap-dim",   # used for logs only; no checkpoints written
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=10,
        save_strategy="no",          # <-- skip ALL local checkpoints (disk-safe)
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        report_to="none",
        # required for vision SFT:
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=1,
        max_length=1024,             # region JSON + summary is longer than a bare caption
    ),
)

print(f"Starting LoRA SFT on SciCap | no local checkpoints | pushing to {HF_REPO} every {PUSH_EVERY} steps")
trainer.train()
print("LoRA SFT complete -- final adapter pushed to:", HF_REPO)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting LoRA SFT on SciCap | no local checkpoints | pushing to shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm every 100 steps


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.069024
20,0.801694
30,0.710058
40,0.752438
50,0.784606
60,0.744449
70,0.684000
80,0.787706
90,0.703039
100,0.750894


...

  ^ pushed adapter to shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm  [step-500]


No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpodbbydk5/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.


  ^ pushed adapter to shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm  [final]
LoRA SFT complete -- final adapter pushed to: shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm


In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted (or use Colab Secrets, see below)

In [ ]:
repo = "shemalfoy/qwen2-vl-scicap-lora-adapter_dim-afterdaptandsft_wutg"   # change to your namespace

model.push_to_hub(repo)        # uploads the LoRA adapter (~tens of MB)
tokenizer.push_to_hub(repo)    # uploads processor/tokenizer config

README.md:   0%|          | 0.00/595 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  93%|#########3|  192MB /  206MB            

Saved model to https://huggingface.co/shemalfoy/qwen2-vl-scicap-lora-adapter_dim-afterdaptandsft


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpajdopiqp/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpajdopiqp/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

In [ ]:
# === Non-misleading 2x2 eval: {baseline, fine-tuned} x {no-context, +context} ===
import numpy as np, torch
from evaluate import load
from PIL import Image

# ---- 0. Held-out set (rows 2000-2009) ----
eval_df = df.iloc[2000:2010].reset_index(drop=True)
eval_ds = [{"image": Image.open(r.image_path).convert("RGB"),
            "caption": str(r.caption)} for _, r in eval_df.iterrows()]

# ---- 1. Contexts via the SAME Feature-1 pipeline (detector -> OCR -> text-LM) ----
SYS = ("You are given the explanatory paragraph printed beneath a scientific "
       "figure. In ONE sentence, state what the figure shows and its key "
       "variables, to help a vision model caption it. No preamble.")

blocks = []
for ex in eval_ds:                                   # _det is still loaded
    ok, box = detect_text_block(ex["image"])
    blk = ""
    if ok:
        crop = _crop_norm(ex["image"], box)
        blk = " ".join(_det.readtext(np.asarray(crop), detail=0, paragraph=True)).strip()
    blocks.append(blk)

# load the text-LM 4-bit so it fits NEXT TO the VL model, use it, free it
from transformers import pipeline, BitsAndBytesConfig
_llm = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                model_kwargs={"quantization_config": BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)},
                device_map="auto")
contexts = []
for blk in blocks:
    if not blk:
        contexts.append(""); continue
    msg = [{"role": "system", "content": SYS},
           {"role": "user", "content": blk[:2000]}]
    contexts.append(_llm(msg, max_new_tokens=60,
                         do_sample=False)[0]["generated_text"][-1]["content"].strip())
del _llm; torch.cuda.empty_cache()
print("contexts found on eval set:", sum(bool(c) for c in contexts), "/", len(contexts))

# ---- 2. Training-style prompt for ALL four arms ----
INSTR = ("First localize each sub-figure as a JSON list of "
         "{\"bbox_2d\":[x1,y1,x2,y2], \"caption\":...}, coords normalized "
         "0-1000, then give an overall 'Summary:' of the whole figure.")

def make_prompt(ctx):
    return f"Context read from the figure: {ctx}\n\n{INSTR}" if ctx else INSTR

def generate_caption(image, prompt):
    messages = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": prompt}]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(image, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=320, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()

def extract_summary(t):                              # score ONLY the summary text
    return t.split("Summary:", 1)[1].strip() if "Summary:" in t else t.strip()

def run_arm(label, use_ctx):
    preds = []
    for i, ex in enumerate(eval_ds):
        p = generate_caption(ex["image"], make_prompt(contexts[i] if use_ctx else ""))
        preds.append(extract_summary(p))
        print(f"  [{label}] {i}: {preds[-1][:60]!r}")
    return preds

FastVisionModel.for_inference(model)
refs = [ex["caption"] for ex in eval_ds]

arms = {}
print("BASELINE (adapters OFF)...")
with model.disable_adapter():
    arms["base / no-ctx"] = run_arm("base-noctx", False)
    arms["base / +ctx"]   = run_arm("base-ctx",   True)
print("FINE-TUNED (adapters ON)...")
arms["ft / no-ctx"]   = run_arm("ft-noctx", False)
arms["ft / +ctx"]     = run_arm("ft-ctx",   True)

# ---- 3. Metrics ----
bleu, rouge = load("sacrebleu"), load("rouge")
meteor, bertscore = load("meteor"), load("bertscore")

def score(preds):
    return {"BLEU":    round(bleu.compute(predictions=preds,
                             references=[[r] for r in refs])["score"], 2),
            "ROUGE-L": round(rouge.compute(predictions=preds,
                             references=refs)["rougeL"] * 100, 2),
            "METEOR":  round(meteor.compute(predictions=preds,
                             references=refs)["meteor"] * 100, 2),
            "BERTScore": round(float(np.mean(bertscore.compute(predictions=preds,
                             references=refs, lang="en")["f1"])) * 100, 2)}

results = {k: score(v) for k, v in arms.items()}
metrics = ["BLEU", "ROUGE-L", "METEOR", "BERTScore"]
hdr = f"{'Arm':<16}" + "".join(f"{m:>11}" for m in metrics)
print("\n" + hdr); print("-" * len(hdr))
for k, m in results.items():
    print(f"{k:<16}" + "".join(f"{m[x]:>11}" for x in metrics))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

contexts found on eval set: 5 / 10
BASELINE (adapters OFF)...


Both `max_new_tokens` (=320) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [base-noctx] 0: '** The figure illustrates the comparison between Leica and S'


...

  [ft-ctx] 8: 'The probability that a random graph has a giant component as'
  [ft-ctx] 9: '— Metallicity vs. mass plane for the galaxies in our sample.'


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Arm                    BLEU    ROUGE-L     METEOR  BERTScore
------------------------------------------------------------
base / no-ctx          4.73      15.88      19.38      83.52
base / +ctx            1.39      13.81      17.32      82.47
ft / no-ctx            0.74      14.79       13.7      85.12
ft / +ctx              1.01      16.72      13.81      85.37


In [ ]:
!pip install -q evaluate sacrebleu rouge_score bert_score nltk

In [ ]:
# ── Baseline vs Fine-tuned eval (SciCap) ────────────────────────────────
import os, torch
import pandas as pd
from PIL import Image
from evaluate import load

FastVisionModel.for_inference(model)

# ---- 1. Build eval_ds from SciCap (rows NOT used in training) ----
csv_path  = "/content/scicap_data/scicap_train.csv"
image_dir = "/content/scicap_data/scicap_images_compressed/share-task-img-mask/arxiv/train"

df = pd.read_csv(csv_path)
df["image"] = df["image"].astype(str).str.strip().map(os.path.basename)
existing   = set(os.listdir(image_dir))
df         = df[df["image"].isin(existing)].copy()
df["image_path"] = image_dir + os.sep + df["image"]

# Training used rows 0-1999 → eval uses rows 2000+ (capped at 100 for speed)
eval_df  = df.iloc[2000:2010].reset_index(drop=True)
eval_ds  = [
    {"image": Image.open(row.image_path).convert("RGB"),
     "caption": str(row.caption)}
    for _, row in eval_df.iterrows()
]
print(f"Eval set: {len(eval_ds)} samples")
assert len(eval_ds) > 0, "eval_ds still empty — check CSV path and image_dir"

# ---- 2. Inference helper ----
def generate_caption(image, prompt="Describe this scientific figure."):
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt},
    ]}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(
        image, input_text, add_special_tokens=False, return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def run_eval(label):
    preds, refs = [], []
    for i, row in enumerate(eval_ds):
        pred = generate_caption(row["image"])
        preds.append(pred)
        refs.append(row["caption"])
        if i % 10 == 0:
            print(f"  [{label}] {i}/{len(eval_ds)}  pred: {pred[:60]!r}")
    return preds, refs

# ---- 3. Baseline (adapters OFF) then fine-tuned (adapters ON) ----
print("Running BASELINE...")
with model.disable_adapter():
    base_preds, refs = run_eval("baseline")

print("\nRunning FINE-TUNED...")
ft_preds, _ = run_eval("fine-tuned")

# ---- 4. Metrics ----
bleu      = load("sacrebleu")
rouge     = load("rouge")
meteor    = load("meteor")
bertscore = load("bertscore")         # note: bertscore, not bert_score

def score(preds, refs):
    return {
        "BLEU":      round(bleu.compute(
                         predictions=preds,
                         references=[[r] for r in refs])["score"], 2),
        "ROUGE-L":   round(rouge.compute(
                         predictions=preds,
                         references=refs)["rougeL"] * 100, 2),
        "METEOR":    round(meteor.compute(
                         predictions=preds,
                         references=refs)["meteor"] * 100, 2),
        "BERTScore": round(sum(bertscore.compute(
                         predictions=preds,
                         references=refs,
                         lang="en")["f1"]) / len(preds) * 100, 2),
    }

base_m = score(base_preds, refs)
ft_m   = score(ft_preds, refs)

print(f"\n{'Metric':<12} {'Baseline':>10} {'Fine-tuned':>12} {'Δ':>8}")
print("-" * 46)
for k in base_m:
    d = round(ft_m[k] - base_m[k], 2)
    arrow = "↑" if d > 0 else ("↓" if d < 0 else "=")
    print(f"{k:<12} {base_m[k]:>10} {ft_m[k]:>12} {d:>+7} {arrow}")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Eval set: 10 samples
Running BASELINE...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

  [baseline] 0/10  pred: 'The provided figure is a graphical representation comparing '


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Metric         Baseline   Fine-tuned        Δ
----------------------------------------------
BLEU               1.35         1.05    -0.3 ↓
ROUGE-L           12.75         14.5   +1.75 ↑
METEOR            21.52        12.62    -8.9 ↓
BERTScore         81.06         83.2   +2.14 ↑


In [ ]:
print(tokenizer.decode(output_ids[0], skip_special_tokens=False))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|

In [ ]:
!du -sh outputs unsloth_compiled_cache outputs/sft_qwen2_scicap/*

12K	outputs
3.4M	unsloth_compiled_cache
du: cannot access 'outputs/sft_qwen2_scicap/*': No such file or directory


In [ ]:
from huggingface_hub import logout
logout()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Not logged in!
